In [100]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
import json

In [101]:
extraction_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
<Role>
You are an expert medical data engineer and clinical annotator.
</Role>

<Task>
Analyze the given medical text chunk and generate a concise contextual header to prepend to it.
</Task>

<Guidelines>
1. **Context Summary:** Start with "This covers {section} where..." (using the given section metadata), then state the specific sub-topic and mechanism described in the chunk.
2. **Factual Integrity:** Do not hallucinate external medical conditions or facts that are not directly supported by or implied within the text chunk.
</Guidelines>

<Example>
<section_metadata>Diagnosis</section_metadata>
<content>
Small pituitary tumors that don't make hormones, called nonfunctioning microadenomas, often don't cause symptoms.
If they are detected, it's typically because of an imaging exam, such as an MRI or a CT scan, that's done for another reason.
Results that show hormone levels are too low need to be followed with other tests, usually imaging exams, to see if a pituitary adenoma may be the cause of those test results.
</content>

[Context: This covers diagnosis where nonfunctioning pituitary microadenomas are often incidentally detected via imaging]
</Example>

<section_metadata>
{section}
</section_metadata>

<content>
{content}
</content>
""",
        )
    ]
)

In [102]:

from pydantic import BaseModel, Field


class ChunkEnrichment(BaseModel):
    context_summary: str = Field(
        description="A concise, one-line summary of the core clinical focus, specific sub-topic, and mechanism described in the chunk (only 10-20 words)."
    )
   



In [103]:

llm=ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.1,
)


agent=llm.with_structured_output(
    ChunkEnrichment)


In [104]:
import json
from pathlib import Path


def enrich_tumor_chunks(tumor_type: str, checkpoint_every: int = 20, limit: int | None = None):
    """
    Enriches tumor chunk JSON files with contextual headers, for any tumor_type.

    tumor_type: e.g. "glioma", "meningioma", "pituitary" — must match the
                filename prefix in tumor_chunks/{tumor_type}_chunks.json
    checkpoint_every: save progress to disk every N processed chunks
    limit: optional cap on number of chunks to process this run (useful for
           testing on a couple of chunks before running the full set).
           Pass None to process all remaining chunks.
    """
    input_path = Path("tumor_chunks") / f"{tumor_type}_chunks.json"
    output_path = Path("enriched_chunks")
    output_path.mkdir(parents=True, exist_ok=True)

    output_file = output_path / f"{tumor_type}_chunks_enriched.json"
    progress_file = output_path / f"{tumor_type}_chunks_progress.json"

    with open(input_path, "r") as f:
        tumor_data = json.load(f)

    # --- RESUME LOGIC ---
    if output_file.exists():
        with open(output_file, "r") as f:
            tumor_data = json.load(f)  # already-enriched data from a prior run

    last_converted_chunk = 0
    if progress_file.exists():
        with open(progress_file, "r") as f:
            last_converted_chunk = json.load(f).get("last_converted_chunk", 0)

    print(f"[{tumor_type}] Resuming from chunk index {last_converted_chunk} / {len(tumor_data)}")

    end_index = len(tumor_data)
    if limit is not None:
        end_index = min(last_converted_chunk + limit, len(tumor_data))

    for i in range(last_converted_chunk, end_index):
        item = tumor_data[i]

        try:
            section = item.get("metadata", {}).get("section", "Unknown")

            enrichment = agent.invoke(
                extraction_prompt.format_messages(
                    content=item["page_content"],
                    section=section,
                )
            ).model_dump()

            item["enrichments"] = enrichment

            header = f"[Context: {enrichment['context_summary']}]"
            item["page_content"] = f"{header}\n{item['page_content']}"

            last_converted_chunk = i + 1

        except Exception as e:
            print(f"[{tumor_type}] Failed at chunk {i}: {e}")
            break

        # --- CHECKPOINT ---
        if (i + 1) % checkpoint_every == 0 or i == end_index - 1:
            with open(output_file, "w") as f:
                json.dump(tumor_data, f, indent=4)

            with open(progress_file, "w") as f:
                json.dump({"last_converted_chunk": last_converted_chunk}, f)

            print(f"[{tumor_type}] Checkpoint saved at chunk {last_converted_chunk}")

    # --- FINAL SAVE ---
    with open(output_file, "w") as f:
        json.dump(tumor_data, f, indent=4)

    with open(progress_file, "w") as f:
        json.dump({"last_converted_chunk": last_converted_chunk}, f)

    print(f"[{tumor_type}] Done. {last_converted_chunk}/{len(tumor_data)} chunks processed.")

    return tumor_data

In [105]:
# meningioma_data = enrich_tumor_chunks("meningioma")

In [ ]:
# pituitary_data = enrich_tumor_chunks("pituitary")

[pituitary] Resuming from chunk index 0 / 46
[pituitary] Checkpoint saved at chunk 20
[pituitary] Checkpoint saved at chunk 40
[pituitary] Checkpoint saved at chunk 46
[pituitary] Done. 46/46 chunks processed.


## 📒 Pinecone Notebook

This part of notebook contains the complete pipeline for building and populating the Pinecone Vector Database:

- Loads scraped medical chunks from `tumor_chunks/`
- Generates dense embeddings using fine-tuned **MiniLM** via HuggingFace
- Fits and saves the **BM25** sparse encoder (`bm25.pkl`)
- Pushes both dense and sparse vectors into **Pinecone** index `tumor-data-fine-tuned-embeddings`
- Configures hybrid search with `dotproduct` metric on `us-east-1` AWS serverless spec

In [ ]:
import torch

In [ ]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:

from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, losses, SentenceTransformerTrainingArguments

In [ ]:
test_model=SentenceTransformer("Gaykar/all-MiniLM-L6-medical-rag")
test_model.to(device)

In [107]:
# from pinecone import Pinecone
import os
from dotenv import load_dotenv
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
import os
from pinecone import Pinecone, ServerlessSpec
from pinecone_text.sparse import BM25Encoder
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import PineconeHybridSearchRetriever
import json
from langchain_core.documents import Document



In [108]:

# with open("TumorWebData.json", "r", encoding="utf-8") as f:
#     data = json.load(f)

# documents = [Document(page_content=d["page_content"], metadata=d["metadata"]) for d in data]


In [109]:
from dotenv import load_dotenv
load_dotenv()

True

In [111]:
TUMOR_CHUNK_FOLDER = r"C:\Users\ATHARVA\Downloads\my codes\web\NeuroAssist\Notebooks\enriched_chunks"
from typing import List

This part contains

In [112]:
def load_documents_from_folder(folder_path):
    documents = []

    for file_name in os.listdir(folder_path):
        if file_name.endswith(".json"):
            file_path = os.path.join(folder_path, file_name)

            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)

                for item in data:
                    documents.append(
                        Document(
                            page_content=item["page_content"],
                            metadata=item["metadata"]
                        )
                    )

    return documents


In [113]:

documents: List[Document] = load_documents_from_folder(TUMOR_CHUNK_FOLDER)

In [114]:
import requests
from langchain_core.embeddings import Embeddings

In [115]:


class MedicalRemoteEmbeddings(Embeddings):
    def __init__(self, endpoint: str):
        self.endpoint = endpoint

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        response = requests.post(f"{self.endpoint}/embed_docs", json={"texts": texts})
        return response.json()["embeddings"]

    def embed_query(self, text: str) -> list[float]:
        response = requests.post(f"{self.endpoint}/embed_query", json={"text": text})
        return response.json()["embedding"]

# Use it in your Hybrid Retriever


In [116]:
embeddings = MedicalRemoteEmbeddings(endpoint="https://gaykar-rag-medical-embeddings.hf.space") 

In [117]:
import os
from pinecone import Pinecone, ServerlessSpec
from pinecone_text.sparse import BM25Encoder
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import PineconeHybridSearchRetriever

In [118]:
PINECONE_API_KEY=os.getenv("PINECONE_API_KEY")

pc=Pinecone(api=PINECONE_API_KEY)
index_name="tumor-data-fine-tuned-embeddings"

In [121]:


PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "neuroassist-data-fine-tuned-embeddings"


# Create index if not exists
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="dotproduct",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print("Index created.")

index = pc.Index(index_name)
print("Index ready:", index.describe_index_stats())






Index created.
Index ready: {'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '154',
                                    'content-type': 'application/json',
                                    'date': 'Mon, 03 Aug 2026 12:05:48 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '45',
                                    'x-pinecone-request-latency-ms': '44',
                                    'x-pinecone-response-duration-ms': '47'}},
 'dimension': 384,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'dotproduct',
 'namespaces': {},
 'storageFullness': 0.0,
 'total_vector_count': 0,
 'vector_type': 'dense'}


In [ ]:
# embeddings = HuggingFaceEmbeddings(
#     model_name="sentence-transformers/multi-qa-MiniLM-L6-cos-v1",
#     model_kwargs={"device": "cpu"})

In [122]:
# )

bm25_encoder = BM25Encoder()

# Fit BM25 on ALL documents
bm25_encoder.fit([doc.page_content for doc in documents])

# -----------------------------------------
# 3. Build Hybrid Retriever
# -----------------------------------------
retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25_encoder,
    index=index
)


100%|██████████| 89/89 [00:00<00:00, 328.57it/s]


In [ ]:

print("\nAdding documents to the index...")

# for doc in documents:
#     retriever.add_texts(
#         texts=[doc.page_content],
#         metadatas=[doc.metadata]
#     )

print("Documents added successfully.")


Adding documents to the index...


100%|██████████| 1/1 [00:02<00:00,  2.03s/it]

Documents added successfully.


In [ ]:
index.describe_index_stats()

In [ ]:
retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25_encoder,
    index=index,
    top_k=5,      # Set your k here
    alpha=0.35     # Optional: 0.0 is pure sparse (BM25), 1.0 is pure dense
)

In [129]:
query = "What are the  causes for glioma ?"

# Simplified call
results = retriever.invoke(query,filter={"tumor_type": "glioma"})

# If you MUST filter dynamically:
# results = retriever.get_relevant_documents(query, filter={"tumor_type": "meningioma"})

print("\n--- Hybrid Search Results ---")
for i, doc in enumerate(results):
    print(f"{i+1}: {doc.page_content[:100]}...")


--- Hybrid Search Results ---
1: [Context: This covers Causes where glioma arises from DNA mutations causing uncontrolled proliferati...
2: [Context: This covers Causes where glioma tumor cells resemble glial cells that support nerve cells....
3: [Context: This covers Overview where glioma is a glial cell tumor that can be benign or malignant, a...
4: [Context: This covers Treatment where targeted therapy blocks glioma cell chemicals to induce cell d...
5: [Context: This covers Treatment where glioma symptoms are managed with anticonvulsants, cognitive en...
